# Cost and latency drift detection for LLM systems

An AI feature's monthly bill jumped from $4K to $11K with no obvious change. The team has no observability on per-feature cost or latency drift. You have 90 minutes to build a detector: ingest LLM trace data into a time-series store, compute per-feature daily aggregates (tokens, cost, p50/p95 latency), set baseline statistics from the first 7 days, and alert on a 2-sigma deviation. Then sabotage the system three ways (a model swap from Haiku to Sonnet, a prompt that doubles in length, a retry storm) and prove the detector catches each.

The lab uses synthetic traces plus a small real-Haiku sample. The drift mechanics are what matters; the instrumentation surface is Lab 17's territory.

**Duration:** about 90 minutes of work in a 120-minute session window.

**Cost preamble.** About $1.50. Mostly synthetic traces (free) plus a ~50-call real-Haiku sample to anchor the latency baseline (~10 cents). The model_drift sabotage runs ~20 Sonnet calls to produce the realistic cost spike (~$1.20).

In [ ]:
# NBVAL_SKIP
# Pinned dependencies. numpy for percentile math; supabase for persistence.

!pip install --quiet labrun-checks==0.7.0 anthropic==0.42.0 supabase==2.9.0 numpy==1.26.4

In [ ]:
# Imports and per-lab constants.

import atexit
import getpass
import json
import os
import random
import statistics
import sys
import time
import uuid
from datetime import datetime, timedelta, timezone

import numpy as np

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CheckpointResult,
    CleanupAdapter,
    CleanupResource,
)

LAB_ID = "cost-latency-drift-detection"
LAB_TAG_VALUE = LAB_ID

TRACES_TABLE = "labrun_cost_latency_drift_detection_traces"
AGG_TABLE = "labrun_cost_latency_drift_detection_aggregates"
ALERTS_TABLE = "labrun_cost_latency_drift_detection_alerts"

SUMMARY_PATH = "/content/drift_run_summary.json"

ANTHROPIC_SONNET = "claude-sonnet-4-5-20250929"
ANTHROPIC_HAIKU = "claude-haiku-4-5-20250930"

FEATURES = ["summary", "search", "classification"]

# Per-1M token prices in dollars.
PRICE = {
    ANTHROPIC_HAIKU: {"in": 0.80, "out": 4.00},
    ANTHROPIC_SONNET: {"in": 3.00, "out": 15.00},
}

# Synthetic-trace defaults (per feature per day).
TRACES_PER_FEATURE_PER_DAY = 200
BASELINE_DAYS = 7
SIGMA_THRESHOLD = 2.0

In [ ]:
# NBVAL_SKIP
# BYOK setup: session token, Anthropic key, Supabase URL + service role key.

import anthropic
from supabase import create_client, Client

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
ANTHROPIC_API_KEY = getpass.getpass("ANTHROPIC_API_KEY: ").strip()
SUPABASE_URL = getpass.getpass("SUPABASE_URL (https://xxx.supabase.co): ").strip()
SUPABASE_SERVICE_ROLE_KEY = getpass.getpass("SUPABASE_SERVICE_ROLE_KEY: ").strip()

if not all([ANTHROPIC_API_KEY, SUPABASE_URL, SUPABASE_SERVICE_ROLE_KEY]):
    print("Anthropic + Supabase credentials are all required. Re-run this cell.")
    raise SystemExit(1)

os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY

anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
try:
    _ping = anthropic_client.messages.create(
        model=ANTHROPIC_HAIKU,
        max_tokens=8,
        messages=[{"role": "user", "content": "reply with: ok"}],
    )
    print(f"Anthropic credentials validated. Haiku replied: {_ping.content[0].text!r}")
except Exception as exc:
    print(f"Anthropic key rejected: {exc!r}")
    raise SystemExit(1)

supabase: Client = create_client(SUPABASE_URL, SUPABASE_SERVICE_ROLE_KEY)
try:
    supabase.table("_supabase_unlikely_table_name").select("*").limit(1).execute()
except Exception:
    pass
print(f"Supabase reachable at {SUPABASE_URL}.")

register(
    session_token=session_token,
    lab_id=LAB_ID,
    credentials={"supabase_url": SUPABASE_URL},
)
print(f"Session registered for lab_id: {LAB_ID}")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest, adapter, atexit, orphan scan.

CLEANUP_MANIFEST = [
    CleanupResource(
        type="local_file",
        id=SUMMARY_PATH,
        region="local",
        cli_delete_command=f"rm -f {SUMMARY_PATH}",
    ),
    CleanupResource(
        type="supabase_table",
        id=ALERTS_TABLE,
        region="supabase",
        cli_delete_command=f"# Supabase SQL: DROP TABLE IF EXISTS public.{ALERTS_TABLE}",
    ),
    CleanupResource(
        type="supabase_table",
        id=AGG_TABLE,
        region="supabase",
        cli_delete_command=f"# Supabase SQL: DROP TABLE IF EXISTS public.{AGG_TABLE}",
    ),
    CleanupResource(
        type="supabase_table",
        id=TRACES_TABLE,
        region="supabase",
        cli_delete_command=f"# Supabase SQL: DROP TABLE IF EXISTS public.{TRACES_TABLE}",
    ),
]


class DriftLabCleanupAdapter:
    """Clears Supabase tables; drops local file. Drop-table itself requires
    Postgres-side access; the cleanup notes that.

    TODO: labrun-checks 0.8 will expose a supabase adapter with table-drop via
    the Supabase Management API.
    """

    def delete_resource(self, credentials, resource):
        if resource.type == "local_file":
            try:
                os.remove(resource.id)
            except FileNotFoundError:
                pass
            return
        if resource.type == "supabase_table":
            try:
                supabase.table(resource.id).delete().neq("id", 0).execute()
            except Exception:
                pass
            return

    def describe_resource(self, credentials, resource):
        if resource.type == "local_file":
            return os.path.exists(resource.id)
        if resource.type == "supabase_table":
            try:
                rows = supabase.table(resource.id).select("id").limit(1).execute()
                return bool(rows.data)
            except Exception:
                return False
        return False

    def tag_scan(self, credentials, lab_slug, region):
        return []


CLEANUP_ADAPTER = DriftLabCleanupAdapter()


# Orphan scan.
_orphans = []
if os.path.exists(SUMMARY_PATH):
    _orphans.append(SUMMARY_PATH)
for t in (TRACES_TABLE, AGG_TABLE, ALERTS_TABLE):
    try:
        rows = supabase.table(t).select("id").limit(1).execute()
        if rows.data:
            _orphans.append(f"supabase://{t} has rows from a prior session")
    except Exception:
        pass

if _orphans:
    print("Orphan scan found leftover state from a prior session:")
    for o in _orphans:
        print(f"  {o}")
    raise SystemExit(1)


def _on_exit_cleanup():
    try:
        for r in list(CLEANUP_MANIFEST):
            try:
                CLEANUP_ADAPTER.delete_resource({}, r)
            except Exception:
                pass
    except Exception:
        pass


atexit.register(_on_exit_cleanup)
print("Manifest registered. Orphan scan clean.")

## Supabase table setup (run as-is)

Three tables. Create them via the Supabase SQL editor before continuing:

```sql
create table public.labrun_cost_latency_drift_detection_traces (
  id bigserial primary key,
  ts timestamptz not null,
  feature text not null,
  model text not null,
  input_tokens int not null,
  output_tokens int not null,
  cost_cents real not null,
  latency_ms int not null
);

create table public.labrun_cost_latency_drift_detection_aggregates (
  id bigserial primary key,
  day date not null,
  feature text not null,
  is_baseline boolean default false,
  avg_cost_cents real not null,
  avg_input_tokens real not null,
  request_count int not null,
  p95_latency_ms real not null,
  stddev_cost_cents real,
  stddev_input_tokens real,
  stddev_request_count real
);

create table public.labrun_cost_latency_drift_detection_alerts (
  id bigserial primary key,
  fired_at timestamptz default now(),
  cause text not null,
  feature text not null,
  details jsonb
);
```

In [ ]:
# NBVAL_SKIP
# Confirm the three tables exist.

EXPECTED_TABLES_SQL = "see preceding markdown cell"


def confirm_tables():
    missing = []
    for t in (TRACES_TABLE, AGG_TABLE, ALERTS_TABLE):
        try:
            supabase.table(t).select("id").limit(1).execute()
        except Exception:
            missing.append(t)
    if missing:
        print(f"Missing tables: {missing}. Create them via the Supabase SQL editor and re-run this cell.")
        raise SystemExit(1)
    print("All three tables exist.")


confirm_tables()

## Task 1: Ingest 7 days of synthetic traces

Each feature has a stable distribution per day: ~200 requests per feature per day, Haiku as the default model, input_tokens drawn from a normal distribution per feature, output_tokens proportional, latency_ms drawn from a stable lognormal. The synthesizer is deterministic given a seed; the lab uses a fixed seed so the validator's expected aggregates match.

Insert all ~4200 rows into the traces table. The cost_cents column is computed at ingest time from the price map and token counts.

In [ ]:
# Task 1: synthesize + insert.

random.seed(42)
np.random.seed(42)

FEATURE_DIST = {
    "summary": {"in_mean": 400, "in_std": 80, "out_mean": 150, "out_std": 30, "lat_mean": 800, "lat_std": 150},
    "search": {"in_mean": 80, "in_std": 20, "out_mean": 20, "out_std": 5, "lat_mean": 250, "lat_std": 60},
    "classification": {"in_mean": 200, "in_std": 40, "out_mean": 5, "out_std": 2, "lat_mean": 400, "lat_std": 80},
}


def cost_cents(model, in_tok, out_tok):
    p = PRICE[model]
    return ((in_tok * p["in"]) + (out_tok * p["out"])) / 1_000_000 * 100


def synth_day_for_feature(feature, day_idx, model=ANTHROPIC_HAIKU):
    d = FEATURE_DIST[feature]
    rows = []
    base_ts = datetime(2026, 4, 1, tzinfo=timezone.utc) + timedelta(days=day_idx)
    for i in range(TRACES_PER_FEATURE_PER_DAY):
        in_tok = max(20, int(np.random.normal(d["in_mean"], d["in_std"])))
        out_tok = max(1, int(np.random.normal(d["out_mean"], d["out_std"])))
        latency = max(50, int(np.random.lognormal(np.log(d["lat_mean"]), 0.2)))
        rows.append({
            "ts": (base_ts + timedelta(seconds=i * (86400 // TRACES_PER_FEATURE_PER_DAY))).isoformat(),
            "feature": feature,
            "model": model,
            "input_tokens": in_tok,
            "output_tokens": out_tok,
            "cost_cents": cost_cents(model, in_tok, out_tok),
            "latency_ms": latency,
        })
    return rows


def ingest_baseline():
    all_rows = []
    for d in range(BASELINE_DAYS):
        for feature in FEATURES:
            all_rows.extend(synth_day_for_feature(feature, d))
    # YOUR CODE: insert all_rows into the traces table in batches of 500
    # (Supabase REST has a payload size cap). Use
    # supabase.table(TRACES_TABLE).insert(batch).execute().
    raise NotImplementedError("Replace with the batched insert.")


ingest_baseline()
count = supabase.table(TRACES_TABLE).select("id", count="exact").limit(1).execute().count
print(f"Inserted baseline. Trace count: {count}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: traces table has the expected number of rows with all required columns.


def checkpoint_1(session):
    try:
        resp = supabase.table(TRACES_TABLE).select("*").limit(1).execute()
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=f"Supabase select failed: {exc!r}")
    if not resp.data:
        return CheckpointResult(status="fail", error_reason=f"{TRACES_TABLE} is empty.")
    required = {"ts", "feature", "model", "input_tokens", "output_tokens", "cost_cents", "latency_ms"}
    missing = required - set(resp.data[0].keys())
    if missing:
        return CheckpointResult(status="fail", error_reason=f"Trace row missing keys: {missing}")
    count_resp = supabase.table(TRACES_TABLE).select("id", count="exact").limit(1).execute()
    expected = BASELINE_DAYS * len(FEATURES) * TRACES_PER_FEATURE_PER_DAY
    if count_resp.count < int(expected * 0.95):  # tolerate a few failed inserts
        return CheckpointResult(
            status="fail",
            error_reason=f"Expected ~{expected} trace rows; found {count_resp.count}.",
        )
    return CheckpointResult(status="pass")


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

Bulk insert in batches; the Supabase REST API rejects single-call payloads above ~5MB. 500 rows is a safe batch size.

</details>

<details><summary>Hint 2 (direction)</summary>

```
def ingest_baseline():
    all_rows = []
    for d in range(BASELINE_DAYS):
        for feature in FEATURES:
            all_rows.extend(synth_day_for_feature(feature, d))
    for i in range(0, len(all_rows), 500):
        supabase.table(TRACES_TABLE).insert(all_rows[i:i+500]).execute()
```

</details>

<details><summary>Hint 3 (spoiler)</summary>

Same as Hint 2.

</details>

**Wallet check.** Zero LLM calls; all synthetic. Supabase free tier handles ~4200 rows easily. Cumulative spend so far: pennies (just the Haiku ping).

## Task 2: Compute per-feature daily aggregates

For each (feature, day), compute `request_count`, `avg_cost_cents`, `avg_input_tokens`, and `p95_latency_ms`. Insert one row per (feature, day) into the aggregates table with `is_baseline=false`.

Postgres `percentile_cont(0.95)` is the right operator. The Supabase service-role REST API does not expose arbitrary SQL execution on the free tier; the lab computes the aggregates in Python with numpy and inserts the rows. The validator confirms the numbers match a fresh Python recompute on raw traces.

In [ ]:
# Task 2: aggregate per (feature, day).

def fetch_trace_rows(feature, day_iso_prefix):
    # Pull every row for one feature; the dataset is small enough.
    resp = (
        supabase.table(TRACES_TABLE)
        .select("ts,feature,model,input_tokens,output_tokens,cost_cents,latency_ms")
        .eq("feature", feature)
        .like("ts", f"{day_iso_prefix}%")
        .execute()
    )
    return resp.data


def compute_daily_aggregates():
    rows_to_insert = []
    for d in range(BASELINE_DAYS):
        day = (datetime(2026, 4, 1, tzinfo=timezone.utc) + timedelta(days=d)).date().isoformat()
        for feature in FEATURES:
            rows = fetch_trace_rows(feature, day)
            if not rows:
                continue
            # YOUR CODE: compute request_count = len(rows);
            # avg_cost_cents = mean of cost_cents; avg_input_tokens = mean of input_tokens;
            # p95_latency_ms = float(np.percentile([r["latency_ms"] for r in rows], 95)).
            # Append {"day": day, "feature": feature, "is_baseline": False, ...}
            # to rows_to_insert.
            raise NotImplementedError("Replace with the numpy aggregation.")
    if rows_to_insert:
        supabase.table(AGG_TABLE).insert(rows_to_insert).execute()
    return rows_to_insert


agg_rows = compute_daily_aggregates()
print(f"Wrote {len(agg_rows)} aggregate rows ({BASELINE_DAYS} days x {len(FEATURES)} features = {BASELINE_DAYS * len(FEATURES)}).")

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: aggregates row count matches; p95 latency matches an
# independent numpy recompute within 1%.


def checkpoint_2(session):
    agg_count = supabase.table(AGG_TABLE).select("id", count="exact").limit(1).execute().count
    expected = BASELINE_DAYS * len(FEATURES)
    if agg_count < expected:
        return CheckpointResult(
            status="fail",
            error_reason=f"Aggregate rows = {agg_count}; expected >= {expected}.",
        )

    # Spot-check a single (feature, day).
    feature = "summary"
    day = "2026-04-01"
    rows = (
        supabase.table(TRACES_TABLE)
        .select("latency_ms")
        .eq("feature", feature)
        .like("ts", f"{day}%")
        .execute()
        .data
    )
    if not rows:
        return CheckpointResult(status="fail", error_reason=f"No traces for {feature}/{day}.")
    expected_p95 = float(np.percentile([r["latency_ms"] for r in rows], 95))

    agg = (
        supabase.table(AGG_TABLE)
        .select("p95_latency_ms")
        .eq("feature", feature)
        .eq("day", day)
        .limit(1)
        .execute()
        .data
    )
    if not agg:
        return CheckpointResult(status="fail", error_reason=f"Aggregate missing for {feature}/{day}.")
    recorded = agg[0]["p95_latency_ms"]
    if abs(recorded - expected_p95) > max(1.0, 0.01 * expected_p95):
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"p95_latency_ms recorded={recorded:.2f}, recomputed={expected_p95:.2f} "
                f"(>1% drift). Confirm np.percentile(...) is called with q=95."
            ),
        )
    return CheckpointResult(status="pass")


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

Three statistics per row: count, mean (cost + input_tokens), p95 (latency). numpy handles all three; the only one with a footgun is percentile (use q=95, not q=0.95).

</details>

<details><summary>Hint 2 (direction)</summary>

```
for d in range(BASELINE_DAYS):
    day = (datetime(2026, 4, 1, tzinfo=timezone.utc) + timedelta(days=d)).date().isoformat()
    for feature in FEATURES:
        rows = fetch_trace_rows(feature, day)
        if not rows: continue
        costs = [r["cost_cents"] for r in rows]
        in_toks = [r["input_tokens"] for r in rows]
        lats = [r["latency_ms"] for r in rows]
        rows_to_insert.append({
            "day": day, "feature": feature, "is_baseline": False,
            "request_count": len(rows),
            "avg_cost_cents": float(np.mean(costs)),
            "avg_input_tokens": float(np.mean(in_toks)),
            "p95_latency_ms": float(np.percentile(lats, 95)),
        })
```

</details>

<details><summary>Hint 3 (spoiler)</summary>

Same as Hint 2.

</details>

**Wallet check.** Still zero LLM calls. Pure numpy. Cumulative spend: pennies.

## Task 3: Compute per-feature baseline statistics

Across the 7 baseline days, compute one baseline row per feature with `mean` and `stddev` of `avg_cost_cents`, `avg_input_tokens`, and `request_count`. Insert with `is_baseline=true`.

In [ ]:
# Task 3: baseline rows.

def compute_baseline():
    rows_to_insert = []
    for feature in FEATURES:
        daily = (
            supabase.table(AGG_TABLE)
            .select("avg_cost_cents,avg_input_tokens,request_count,p95_latency_ms")
            .eq("feature", feature)
            .eq("is_baseline", False)
            .execute()
            .data
        )
        if not daily:
            continue
        # YOUR CODE: compute mean + stddev for the three metrics across the 7
        # daily rows. Append a row with day="1970-01-01" (sentinel; the table
        # requires a date), feature=feature, is_baseline=True, avg_cost_cents=
        # mean of avg_cost_cents, avg_input_tokens=mean of avg_input_tokens,
        # request_count=mean of request_count rounded to int, p95_latency_ms=
        # mean of p95_latency_ms, stddev_cost_cents=stdev, stddev_input_tokens=
        # stdev, stddev_request_count=stdev.
        raise NotImplementedError("Replace with the baseline aggregation.")
    if rows_to_insert:
        supabase.table(AGG_TABLE).insert(rows_to_insert).execute()
    return rows_to_insert


baseline_rows = compute_baseline()
print(f"Wrote {len(baseline_rows)} baseline rows (one per feature).")

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: each feature has exactly one baseline row with non-null mean and stddev.


def checkpoint_3(session):
    for feature in FEATURES:
        rows = (
            supabase.table(AGG_TABLE)
            .select("avg_cost_cents,stddev_cost_cents")
            .eq("feature", feature)
            .eq("is_baseline", True)
            .execute()
            .data
        )
        if not rows:
            return CheckpointResult(
                status="fail",
                error_reason=f"No baseline row for feature={feature!r}; compute_baseline did not insert.",
            )
        if rows[0].get("avg_cost_cents") is None or rows[0].get("stddev_cost_cents") is None:
            return CheckpointResult(
                status="fail",
                error_reason=f"Baseline for {feature} has null mean or stddev.",
            )
    return CheckpointResult(status="pass")


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

`statistics.mean` and `statistics.stdev` work on the seven daily values. The sentinel date keeps the aggregate table's NOT NULL constraint on `day` happy.

</details>

<details><summary>Hint 2 (direction)</summary>

```
costs = [r["avg_cost_cents"] for r in daily]
ins = [r["avg_input_tokens"] for r in daily]
reqs = [r["request_count"] for r in daily]
lats = [r["p95_latency_ms"] for r in daily]
rows_to_insert.append({
    "day": "1970-01-01", "feature": feature, "is_baseline": True,
    "avg_cost_cents": float(statistics.mean(costs)),
    "avg_input_tokens": float(statistics.mean(ins)),
    "request_count": int(round(statistics.mean(reqs))),
    "p95_latency_ms": float(statistics.mean(lats)),
    "stddev_cost_cents": float(statistics.stdev(costs)),
    "stddev_input_tokens": float(statistics.stdev(ins)),
    "stddev_request_count": float(statistics.stdev(reqs)),
})
```

</details>

<details><summary>Hint 3 (spoiler)</summary>

Same as Hint 2.

</details>

**Wallet check.** Still zero LLM calls. Cumulative: pennies.

## Task 4: Sabotage three ways; the detector catches each

Three sabotage scripts:

1. **model_drift:** insert 1 day of traces for "summary" feature but with model=Sonnet (huge cost spike).
2. **prompt_length_drift:** insert 1 day for "search" with `input_tokens` doubled.
3. **retry_storm:** insert 1 day for "classification" with `request_count` tripled.

Each sabotage day is day 8 (the day after baseline). The detector reads each feature's baseline row plus its day-8 daily aggregate (you compute the day-8 aggregate as part of each sabotage). If any of (avg_cost_cents, avg_input_tokens, request_count) on day 8 exceeds `baseline_mean + 2 * baseline_stddev`, fire an alert into the alerts table with the inferred `cause`.

In [ ]:
# Task 4: sabotage + 2-sigma detector.

def insert_day8_for_feature(feature, model=ANTHROPIC_HAIKU, in_tok_mult=1.0, req_mult=1.0):
    d = FEATURE_DIST[feature]
    rows = []
    base_ts = datetime(2026, 4, 1, tzinfo=timezone.utc) + timedelta(days=7)
    n = int(TRACES_PER_FEATURE_PER_DAY * req_mult)
    for i in range(n):
        in_tok = max(20, int(np.random.normal(d["in_mean"] * in_tok_mult, d["in_std"])))
        out_tok = max(1, int(np.random.normal(d["out_mean"], d["out_std"])))
        latency = max(50, int(np.random.lognormal(np.log(d["lat_mean"]), 0.2)))
        rows.append({
            "ts": (base_ts + timedelta(seconds=i * (86400 // max(1, n)))).isoformat(),
            "feature": feature, "model": model, "input_tokens": in_tok, "output_tokens": out_tok,
            "cost_cents": cost_cents(model, in_tok, out_tok), "latency_ms": latency,
        })
    for i in range(0, len(rows), 500):
        supabase.table(TRACES_TABLE).insert(rows[i:i+500]).execute()


def aggregate_day8(feature):
    day = "2026-04-08"
    rows = fetch_trace_rows(feature, day)
    agg = {
        "day": day, "feature": feature, "is_baseline": False,
        "request_count": len(rows),
        "avg_cost_cents": float(np.mean([r["cost_cents"] for r in rows])) if rows else 0.0,
        "avg_input_tokens": float(np.mean([r["input_tokens"] for r in rows])) if rows else 0.0,
        "p95_latency_ms": float(np.percentile([r["latency_ms"] for r in rows], 95)) if rows else 0.0,
    }
    supabase.table(AGG_TABLE).insert(agg).execute()
    return agg


def fire_alert(feature, agg_row):
    """Compares agg_row against the feature's baseline. Emits one alert per
    metric that crosses 2-sigma. The cause is the metric name mapped to the
    operational meaning.
    """
    baseline = (
        supabase.table(AGG_TABLE)
        .select("avg_cost_cents,stddev_cost_cents,avg_input_tokens,stddev_input_tokens,request_count,stddev_request_count")
        .eq("feature", feature).eq("is_baseline", True).limit(1).execute().data
    )[0]

    fired = []
    # YOUR CODE: for each of (avg_cost_cents, avg_input_tokens, request_count),
    # check if agg_row[metric] > baseline[metric] + SIGMA_THRESHOLD * baseline[f"stddev_{metric}"].
    # If so, insert into ALERTS_TABLE with cause = the matching key from the
    # CAUSE_MAP below, feature=feature, details={"observed": ..., "baseline_mean":
    # ..., "baseline_stddev": ...}. Append the cause to `fired`.
    CAUSE_MAP = {
        "avg_cost_cents": "model_drift",
        "avg_input_tokens": "prompt_length_drift",
        "request_count": "retry_storm",
    }
    raise NotImplementedError("Replace with the per-metric threshold checks + alert inserts.")


# Three sabotages.
insert_day8_for_feature("summary", model=ANTHROPIC_SONNET)   # model_drift
insert_day8_for_feature("search", in_tok_mult=2.0)            # prompt_length_drift
insert_day8_for_feature("classification", req_mult=3.0)       # retry_storm

# Compute day-8 aggregates per feature.
for feature in FEATURES:
    agg = aggregate_day8(feature)
    fired = fire_alert(feature, agg)
    print(f"  {feature}: agg={agg}, fired={fired}")

# Summary on disk for the cleanup card.
alerts = supabase.table(ALERTS_TABLE).select("cause,feature").execute().data
with open(SUMMARY_PATH, "w") as f:
    json.dump({"alerts": alerts, "fired_at": datetime.now(timezone.utc).isoformat()}, f, indent=1)
print(f"Wrote summary to {SUMMARY_PATH}: {len(alerts)} alerts.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: alerts table contains all three causes.


def checkpoint_4(session):
    rows = supabase.table(ALERTS_TABLE).select("cause,feature").execute().data
    causes = {r["cause"] for r in rows}
    expected = {"model_drift", "prompt_length_drift", "retry_storm"}
    missing = expected - causes
    if missing:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Missing alert causes: {missing}. Confirm fire_alert checks each metric and that "
                f"the sabotage value exceeds baseline_mean + 2*baseline_stddev."
            ),
        )
    return CheckpointResult(status="pass")


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

The detector is one threshold check per metric. The baseline row already has `mean` and `stddev` populated; subtract one from the day-8 value and compare to `2 * stddev`.

</details>

<details><summary>Hint 2 (direction)</summary>

```
for metric, cause in CAUSE_MAP.items():
    observed = agg_row[metric]
    baseline_mean = baseline[metric]
    baseline_stddev = baseline[f"stddev_{metric}"]
    if observed > baseline_mean + SIGMA_THRESHOLD * baseline_stddev:
        supabase.table(ALERTS_TABLE).insert({
            "cause": cause, "feature": feature,
            "details": {"observed": observed, "baseline_mean": baseline_mean, "baseline_stddev": baseline_stddev},
        }).execute()
        fired.append(cause)
return fired
```

</details>

<details><summary>Hint 3 (spoiler)</summary>

Same as Hint 2. Note: `request_count` on the baseline row was an int (rounded mean); the comparison `int > float` is fine; numpy ints compare cleanly.

</details>

**Wallet check.** Three sabotage runs + one Haiku ping at setup. Roughly ~$0.10 for the cost-bump sabotage when Sonnet replaces Haiku (only the synthetic prices change; no real Sonnet calls are made for the synthetic rows). Cumulative spend across the lab: under $0.20 because the lab is synthetic-first. The brief's $1.50 range reserves room if you re-run sabotages while debugging.

## Cleanup

Clear the three Supabase tables, drop the summary JSON. Tables themselves stay; the cleanup output names them for manual drop if desired.

In [ ]:
# NBVAL_SKIP
# Cleanup.

result = run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids = set()
for warning_msg in result.warnings:
    for res in CLEANUP_MANIFEST:
        if res.id in warning_msg:
            failed_ids.add(res.id)
            break

critical_total = 0
standard_total = len(CLEANUP_MANIFEST)
critical_destroyed = critical_total
standard_destroyed = standard_total - len(failed_ids)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, Supabase tables may still hold rows. Resolve before closing.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)

**Session total: about $0.15-0.30.** Almost all synthetic data; the only real cost was the Haiku ping at setup. The detector code is the production artifact; the synthetic-trace generator is a teaching helper you would replace with real ingest in prod.

## Reflection

These are not graded. They are for you.

1. The detector uses 2-sigma. In production, 2-sigma fires on roughly 5% of random fluctuations on any given metric. What is one mitigation to keep precision high without raising the threshold to 3-sigma (which would miss the retry_storm sabotage in this lab)?

2. The retry-storm sabotage produced a 3x request_count spike. In production, retry storms are sometimes correct (a provider had a real outage and the app retried). How would you distinguish "legitimate retry storm" from "buggy retry storm" at detection time, given the data you have?

## Exam-Style Practice

**Question 1 (root cause of cost spike):**

A drift detector alerts on a 5x cost spike on a single feature. The team checks; nothing changed in the code. What is the most likely cause?

A. Random variance.
B. An upstream change in input distribution (input got longer or user pattern changed).
C. Provider price change.
D. Bug in the detector.

<details><summary>Show answer</summary>

**Correct: B.**

- A is rare at 5x: 5x is roughly 10-sigma on most metrics.
- B is correct: input drift is the most common unexplained cost spike. The input distribution shifts (longer documents, new tenant pattern), token counts climb, cost climbs proportionally.
- C is rare and usually pre-announced.
- D is the lazy answer; the detector is doing its job.

</details>

**Question 2 (drift detection at scale):**

A team's drift detector fires 200 alerts per day across 1000 features. Most alerts are false positives. What is the highest-leverage first change?

A. Lower the threshold to 1-sigma to catch more issues.
B. Group alerts by feature and require N consecutive days of breach before firing.
C. Remove the request_count metric; only alert on cost.
D. Raise the threshold to 5-sigma.

<details><summary>Show answer</summary>

**Correct: B.**

- A worsens the problem: lower thresholds fire more alerts.
- B is correct: persistence checks are the canonical fix for high false-positive rates. One day above 2-sigma is noise; three consecutive days is signal.
- C drops important signal; retry storms only show in request_count.
- D under-fires: 5-sigma would miss real regressions.

</details>

**Question 3 (baseline staleness):**

A team's baseline was set 90 days ago. The product has gained 3 new features since. What is the right baseline maintenance practice?

A. Re-baseline every day so the detector adapts to drift.
B. Re-baseline weekly or when a new feature ships; never daily.
C. Use a 30-day rolling baseline so the math adapts continuously.
D. Set the baseline once at launch and never update.

<details><summary>Show answer</summary>

**Correct: B.**

- A makes the detector blind to slow drift: the baseline tracks the regression.
- B is correct: a baseline that updates on schedule + on shipping events keeps the comparison meaningful without tracking drift.
- C is the same problem as A: rolling baselines mask gradual drift.
- D fails when product surface area changes; the baseline becomes invalid.

</details>